# HMApp — inspect a completed Raiders experiment

Change **RUN_DIRECTORY** below and run all cells. This notebook only reads completed
artifacts. It does not import BERT, generate embeddings, download resources or require a GPU.
The original ranking is preserved; the additional views are inspection filters, not improved
performance claims. Charts use plain HTML, like the reference-preparation notebook, so no
new plotting dependency or internet connection is needed.

## 1. Run overview

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src/utils/results.py").is_file():
    raise RuntimeError("Open this notebook from the repository root or src directory.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from utils.results import load_run, inspection_view, phrase_pattern, highlight_word_spans, bar_chart_html

RUN_DIRECTORY = PROJECT_ROOT / "artifacts/raiders_antisemitism_broad_seeds"

run_info, tables, candidate_stage_counts = load_run(RUN_DIRECTORY)
posts = tables["discovery_posts"]
rankings = tables["ranked_candidates"]
occurrences = tables["candidate_occurrences"]
post_scores = tables["candidate_post_scores"]
seed_words = run_info["seed_words"]
anchor_variants = list(dict.fromkeys(
    variant for variants in run_info["anchor_terms"].values() for variant in variants
))

display(pd.Series({
    "run": str(RUN_DIRECTORY), "started_utc": run_info["started_at_utc"],
    "source": run_info["retrieval"]["source_path"],
    "reference_bank": run_info["reference_bank"]["path"],
    "reference_bank_id": run_info["reference_bank"]["identifier_sha256"],
    "checkpoint": run_info["embedding_config"]["checkpoint"],
    "model_revision": run_info["embedding_config"]["resolved_revision"],
    "context_tokens_each_side": run_info["embedding_config"]["context_tokens_each_side"],
    "batch_size": run_info["candidate_batch_size"], "device": run_info["device"],
    "calculation_seconds": run_info["calculation_seconds"],
}))
print("Retrieval seeds:", seed_words)
print("Seed provenance:", run_info["seed_provenance"])
print("Separation:", run_info["retrieval"]["separation"])
display(pd.Series(run_info["stage_seconds"], name="seconds").to_frame())
reference_coverage = tables["reference_coverage"].set_index("anchor")
display(reference_coverage)
display(HTML(bar_chart_html(reference_coverage["reference_vectors"].sort_values(ascending=False),
                            "Fixed reference vectors per anchor")))

## 2. Retrieved corpus

Counts refer to the consumed input prefix, not necessarily the whole source file.
Excluded reference-thread posts are counted without parsing their text. Copied-text exclusions
apply to the remaining examined posts. A seed match indicates retrieval relevance, not a hate label.
The full raw source post is retained in `raw_post` for later work; no conversation is reconstructed.

In [ ]:
display(pd.Series(run_info["retrieval"]["counts"]).to_frame("count"))
print("Stopped at:", run_info["retrieval"]["stop_reason"])
print("Selection:", run_info["retrieval"]["selection"])
display(tables["retrieval_summary"])
display(HTML(bar_chart_html(tables["retrieval_summary"].set_index("seed")["selected_posts"],
                            "Selected posts per retrieval seed (counts can overlap)")))
post_lengths = posts["text"].str.len()
display(post_lengths.describe().to_frame("text_characters"))
display(HTML(bar_chart_html(pd.cut(post_lengths, bins=12).value_counts(sort=False),
                            "Post lengths in characters: counts per interval")))
post_dates = pd.to_datetime(pd.to_numeric(posts["timestamp"], errors="coerce"), unit="s", utc=True)
print("UTC coverage:", post_dates.min(), "to", post_dates.max())
print("Posts with unknown timestamp:", int(post_dates.isna().sum()))
with pd.option_context("display.max_colwidth", 240):
    display(posts[["post_key", "thread_key", "timestamp", "matched_seeds", "text"]].head())

## 3. Candidate generation

The existing baseline removes stopwords **before** constructing n-grams. A candidate such as
`generations holocaust` can therefore span `generations after the Holocaust` in the source.
This run retains that behavior and flags such occurrences instead of silently replacing the algorithm.
Word spans address the exact saved text. TF-IDF selection remains strictly above the mean of
each term's maximum TF-IDF across posts; the POS rules remain the existing provisional rules.

In [ ]:
ngram_counts = pd.DataFrame.from_dict(candidate_stage_counts["ngrams"], orient="index")
ngram_counts.index.name = "ngram_length"
display(ngram_counts)
display(pd.Series({key: value for key, value in candidate_stage_counts.items() if key != "ngrams"}))
tfidf_candidates = tables["tfidf_candidates"]
display(tfidf_candidates.groupby("ngram_length").agg(
    terms=("candidate", "size"), selected=("selected", "sum")
))
display(HTML(bar_chart_html(tfidf_candidates.loc[tfidf_candidates["selected"], "ngram_length"]
                            .value_counts().sort_index(), "TF-IDF-selected candidates by n-gram length")))
print("TF-IDF threshold:", run_info["tfidf_threshold"])
display(tfidf_candidates.sort_values("max_tfidf", ascending=False).head(20))
noncontiguous_occurrences = occurrences.loc[~occurrences["is_contiguous"]]
with pd.option_context("display.max_colwidth", 160):
    display(noncontiguous_occurrences[["candidate", "matched_text", "post_key", "word_spans"]].head(15))

## 4. Semantic ranking and inspection views

Each candidate/post score averages five nearest reference vectors; several can represent the
same anchor. Candidate relevance averages those scores across source posts. The relevance cutoff
is the mean candidate score. Singletons remain in the main ranking.

Seed/anchor exclusion matches whole phrases with word boundaries, ignoring case; it does not
remove unrelated substrings. `original_rank` always refers to the full ranking.
`display_position` describes a filtered view only. These views do not rescore candidates.

In [ ]:
print("HMApp relevance cutoff:", run_info["relevance_threshold"])
print("Above cutoff:", int(rankings["passes_relevance"].sum()), "/", len(rankings))
display(HTML(bar_chart_html(pd.cut(rankings["relevance"], bins=20).value_counts(sort=False),
                            f'Relevance distribution; cutoff = {run_info["relevance_threshold"]:.6f}')))
display(HTML(bar_chart_html(pd.cut(rankings["n_posts"], bins=[0, 1, 2, 5, 10, float("inf")])
                            .value_counts(sort=False), "Distinct scored posts per candidate")))
display(rankings.head(30))

In [ ]:
seed_excluded_view = inspection_view(rankings, excluded_phrases=seed_words)
anchor_excluded_view = inspection_view(rankings, excluded_phrases=anchor_variants)
two_post_view = inspection_view(rankings, minimum_posts=2)
two_thread_view = inspection_view(rankings, minimum_threads=2)

for view_name, view in [
    ("Without seeds or seed-containing phrases", seed_excluded_view),
    ("Without anchors or anchor-containing phrases", anchor_excluded_view),
    ("At least two scored posts", two_post_view),
    ("At least two source threads", two_thread_view),
]:
    print(view_name, f"({len(view):,} candidates)")
    display(view.head(20))

## 5. Inspect an individual candidate

Change `selected_term` to a candidate from the tables. The highlights use the saved constituent
word offsets, so a noncontiguous candidate highlights its words rather than pretending the
intervening text is part of the extracted expression. The preview shows one occurrence per
supporting post, up to five posts; the occurrence table preserves all occurrences.

In [ ]:
selected_term = rankings.iloc[0]["candidate"]  # Replace with a candidate string to inspect.
selected_ranking = rankings.loc[rankings["candidate"].eq(selected_term)]
if selected_ranking.empty:
    raise ValueError(f"Candidate not found in this run: {selected_term!r}")
display(selected_ranking)
selected_occurrences = occurrences.loc[occurrences["candidate"].eq(selected_term)]
display(selected_occurrences[["post_key", "thread_key", "matched_text", "start", "end", "is_contiguous"]])
source_posts = posts.set_index("post_key")
selected_post_scores = post_scores.loc[post_scores["candidate"].eq(selected_term)].set_index("post_key")
for occurrence in selected_occurrences.drop_duplicates("post_key").head(5).itertuples(index=False):
    source_post = source_posts.loc[occurrence.post_key]
    print("Post:", occurrence.post_key, "Thread:", occurrence.thread_key,
          "Relevance:", selected_post_scores.loc[occurrence.post_key, "relevance"])
    display(HTML(highlight_word_spans(source_post["text"], occurrence.word_spans)))

## 6. Summary diagnostics for the report

These are descriptive observations, not precision/recall/F1 or proof of hateful meaning.
Distinct posts and threads are proxies for support, not proof of independent adoption.
Reference coverage concerns the frozen bank; missing anchors are not filled from discovery.

In [ ]:
top_candidates = rankings.head(50)
seed_pattern = phrase_pattern(seed_words)
anchor_pattern = phrase_pattern(anchor_variants)
diagnostics = pd.Series({
    "ranked_candidates": len(rankings),
    "singleton_candidates": int(rankings["n_posts"].eq(1).sum()),
    "singleton_fraction": float(rankings["n_posts"].eq(1).mean()),
    "top_50_actual_size": len(top_candidates),
    "top_50_containing_seeds": int(top_candidates["candidate"].map(lambda text: bool(seed_pattern.search(text))).sum()),
    "top_50_containing_anchors": int(top_candidates["candidate"].map(lambda text: bool(anchor_pattern.search(text))).sum()),
    "anchors_represented": int(reference_coverage["reference_vectors"].gt(0).sum()),
    "anchors_in_vocabulary": len(reference_coverage),
    "noncontiguous_occurrences": len(noncontiguous_occurrences),
    "candidates_with_noncontiguous_occurrences": int(rankings["n_noncontiguous_occurrences"].gt(0).sum()),
})
display(diagnostics.to_frame("value"))
for limitation in run_info["limitations"]:
    print("-", limitation)

### Notes to write after inspecting the actual outputs

- Which candidates appear meaningful in their original posts? Which look malformed?
- How much does the visible list change after hiding known seeds/anchors, while scores stay fixed?
- How much support comes from multiple posts and multiple threads?
- Which missing or dominant anchors could limit interpretation?

Record examples and uncertainties. A high cosine score alone does not establish hateful usage
or that an expression is historically new. Evaluation and conversational-context methods are separate tasks.